In [ ]:
%pip install "/lakehouse/default/Files/shared_libraries/python_utils-0.1.0-py3-none-any.whl"
import base64, yaml, asyncio, logging, threading

from datetime import datetime
from typing import Any, Dict, List, Mapping

from python_utils.http_client import AsyncApiClient, HttpClientSettings
from python_utils.lakehouse_utils import save_json_to_lakehouse
from python_utils.logging_utils import configure_logging

In [ ]:
# ## For local test, uncomment all this cell-block to run maunally the notebook and use test parameters string

# param_string_b64 = 'dummy'
# test_config_yaml = """
# source_system: JSONplaceholder
# authentication_method: api_key
# akv_secret: placeholder_api_key
# base_url: https://jsonplaceholder.typicode.com
# execution_date: "2026-05-15" 
# data_interval_start: "2026-05-12"
# data_interval_end: "2026-05-15"
# api_rate_limit: 2

# tables:
#   - source_table: posts
#     relative_url: /workobject/posts
#     load_type: merge
#     merge_columns: [postId]
#     activated: true
#     pagination_mode: offset
#     page_size: 100
#     batch_key: value
#     incremental_column: LastModifiedDateTime
#     incremental_strategy: odata_filter
#     orderby:  LastModifiedDateTime asc, postId asc
#     select_columns: [
#       postId, CreateDateTime, Category, CreatedBy, Priority, Service, Urgency, Symptom, LastModifiedDateTime
#       ]
# """
# param_string_b64 = base64.urlsafe_b64encode(test_config_yaml.encode('utf-8')).decode('utf-8')

In [ ]:
# Decode the parameter
decoded_param_string_b64 = base64.urlsafe_b64decode(param_string_b64.encode('utf-8')).decode('utf-8') 
CONFIG_YAML = yaml.safe_load(decoded_param_string_b64)


# Set base variables
source_system = (CONFIG_YAML.get('source_system') or '').lower()
akv_secret_key = CONFIG_YAML.get('akv_secret') or CONFIG_YAML.get('akv_sectret')
base_url = CONFIG_YAML.get('base_url')
api_rate_limit = CONFIG_YAML.get('api_rate_limit')
execution_date = CONFIG_YAML.get('execution_date')
data_interval_start = CONFIG_YAML.get('data_interval_start')
data_interval_end = CONFIG_YAML.get('data_interval_end')

execution_datetime = datetime.fromisoformat(execution_date)
execution_date_iso = execution_datetime.isoformat()
execution_date_str = execution_datetime.strftime("%Y-%m-%d")

data_interval_start_dt = datetime.fromisoformat(data_interval_start)
data_interval_end_dt = datetime.fromisoformat(data_interval_end)
data_interval_start_iso = data_interval_start_dt.isoformat()
data_interval_end_iso = data_interval_end_dt.isoformat()



# Retrieve client and client_secret from Key Vault
CLIENT_SECRET = notebookutils.credentials.getSecret('https://kv-analytics-fabric-dev.vault.azure.net/', akv_secret_key)

In [ ]:
"""Functions for extracting JSONplaceholder data via the REST API and OFFSET."""

logger = logging.getLogger(__name__)


def build_client_settings(config: Mapping[str, Any]) -> HttpClientSettings:
    rps = float(config.get("api_rate_limit", 2))

    return HttpClientSettings(
        base_url=str(config["base_url"]).rstrip("/"),
        enable_http2=False,
        max_connections=10,
        max_keepalive=10,
        keepalive_expiry=15.0,
        connect_timeout=10.0,
        read_timeout=60.0,
        concurrency=4,
        rps=rps,
        burst=max(int(rps), 1),
        default_headers={
            "Authorization": f"rest_api_key={CLIENT_SECRET}",
            "Accept": "application/json",
        },
    )

def build_odata_filter(table_cfg: Mapping[str, Any]) -> str:
    incremental_column = table_cfg.get("incremental_column", "LastModifiedDateTime")

    start_value = data_interval_start_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    end_value = data_interval_end_dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    return (
        f"{incremental_column} ge {start_value} and "
        f"{incremental_column} lt {end_value}"
    )

def should_use_incremental_filter(table_cfg: Mapping[str, Any]) -> bool:
    load_type = str(table_cfg.get("load_type", "")).lower()
    incremental_strategy = str(table_cfg.get("incremental_strategy", "")).lower()

    return load_type == "merge" and incremental_strategy == "odata_filter"

async def fetch_JSONplaceholder_table(
    api: AsyncApiClient,
    table_cfg: Mapping[str, Any],
) -> List[Dict[str, Any]]:
    source_table = str(table_cfg.get("source_table", "unknown"))
    endpoint = str(table_cfg["relative_url"]).lstrip("/")
    page_size = int(table_cfg.get("page_size", 100))
    offset = 0
    page_no = 1
    all_rows: List[Dict[str, Any]] = []

    params: Dict[str, Any] = {
        "$top": page_size,
        "$orderby": table_cfg.get("orderby", "LastModifiedDateTime asc, RecId asc"),
    }

    # load_type: merge ➜ Apply OData filter for delta-load, incremental-load
    # load_type: full ➜ fetch ignores $filter, initial-load with pagination across the entire table, full-load
    if should_use_incremental_filter(table_cfg):
        params["$filter"] = build_odata_filter(table_cfg)

    select_columns = table_cfg.get("select_columns")
    if select_columns:
        params["$select"] = ",".join(select_columns)

    logger.info(
        "START table=%s execution_date=%s interval=[%s -> %s] page_size=%s filter=%s selected_columns=%s endpoint=%s",
        source_table,execution_date_iso,data_interval_start_iso,data_interval_end_iso,page_size,params.get("$filter"),len(select_columns or []),endpoint,
    )

    while True:
        params["$skip"] = offset

        response = await api._request("GET", endpoint, params=params)
        # catch 204 No Content and treat it as 0 rows.
        if response.status_code == 204:
            logger.info(
                "END table=%s rows=%s pages=%s execution_date=%s interval=[%s -> %s]",
                source_table,len(all_rows),page_no - 1 if page_no > 1 else 0,execution_date_iso,data_interval_start_iso,data_interval_end_iso,
            )
            break

        payload = response.json()
        rows = payload.get("value", [])

        if not isinstance(rows, list):
            raise ValueError(
                f"Expected list in response.value for table {source_table}"
            )

        fetched_count = len(rows)
        all_rows.extend(rows)

        logger.info(
            "PAGE table=%s page=%s skip=%s fetched=%s total=%s",
            source_table,page_no,offset,fetched_count,len(all_rows),
        )

        if fetched_count < page_size:
            logger.info(
                "END table=%s rows=%s pages=%s execution_date=%s interval=[%s -> %s]",
                source_table,len(all_rows),page_no,execution_date_iso,data_interval_start_iso,data_interval_end_iso,
            )
            break

        offset += page_size
        page_no += 1

    return all_rows

def build_metadata(table_cfg: Mapping[str, Any], record_count: int) -> Dict[str, Any]:
    return {
        "source_system": source_system,
        "source_table": table_cfg.get("source_table"),
        "relative_url": table_cfg.get("relative_url"),
        "resolved_endpoint": f"{base_url.rstrip('/')}/{str(table_cfg.get('relative_url')).lstrip('/')}",
        "load_type": table_cfg.get("load_type"),
        "pagination_mode": table_cfg.get("pagination_mode"),
        "page_size": table_cfg.get("page_size"),
        "records_returned": record_count,
        "execution_date": execution_date_iso,
        "data_interval_start": data_interval_start_iso,
        "data_interval_end": data_interval_end_iso,
        "extraction_method": "JSONplaceholder_odata_api_key_request",
        "api_base_url": base_url,
        "api_rate_limit": api_rate_limit,
        "incremental_column": table_cfg.get("incremental_column"),
    }


async def run() -> None:
    configure_logging()

    config = CONFIG_YAML
    settings = build_client_settings(config)
    api = AsyncApiClient(settings)

    # Choose tables that are active in yml
    activated_tables = [
        t for t in config.get("tables", [])
        if t.get("activated", True)
    ]

    logger.info(
        "JSONplaceholder EXTRACT START execution_date=%s interval=[%s -> %s] tables=%s",execution_date_iso,data_interval_start_iso,data_interval_end_iso,len(activated_tables),
    )

    try:
        for table_cfg in activated_tables:
            source_table = table_cfg.get("source_table")
            if not source_table:
                raise ValueError("Missing source_table in table config")

            rows = await fetch_JSONplaceholder_table(api, table_cfg)
            metadata = build_metadata(table_cfg, len(rows))

            save_json_to_lakehouse(
                rows,
                metadata,
                source_system,
                source_table.lower(),
                execution_date_str,
            )

            logger.info(
                "SAVED table=%s rows=%s target=Files/landing/%s/%s/%s",source_table,len(rows),source_system,source_table.lower(),execution_date_str,
            )

        logger.info(
            "JSONplaceholder EXTRACT END execution_date=%s interval=[%s -> %s]",execution_date_iso,data_interval_start_iso,data_interval_end_iso,
        )

    finally:
        await api.aclose()

In [ ]:
""" Execution cell """

def _launch_run() -> None:

    try:
        asyncio.get_running_loop()
    except RuntimeError:
        asyncio.run(run())
        return

    exception_holder: Dict[str, BaseException] = {}

    def _runner() -> None:
        try:
            asyncio.run(run())
        except BaseException as exc:
            exception_holder["exception"] = exc

    thread = threading.Thread(target=_runner, name="JSONplaceholder-extract-runner", daemon=False)
    thread.start()
    thread.join()

    if exc := exception_holder.get("exception"):
        raise exc   

if __name__ == "__main__":
    _launch_run()